# โน้ตบุ๊กของ day20_bestagon

Day 20 · Hexagons — "Hexagons, the good old bestagons."

โจทย์ (ดึงจาก 30daymapchallenge.com อ่านสด 23:05 น. ของวันที่ 20 พ.ย.) ท้าให้รักหกเหลี่ยม
ผมจึงตรวจสุขภาพมัน: หกเหลี่ยม H3 res 5 ที่ซีรีส์นี้ใชามาตั้งแต่ Day 3 *เป็นหกเหลี่ยมจริงแค่ไหน*

วัดทุกเซลล์ของกริด Day 3 (1,738 เซลล์) ใน projection Equidistant Conic (หน่วยเมตร)
  • compactness = 4πA/P²  (วงกลม = 1, หกเหลี่ยมด้านเท่ามุมเท่า = π√3/6 = 0.9069)
  • จำนวนชิ้น (Polygon vs MultiPolygon) และจำนวนจุดมุม
  • ความไม่สม่ำเสมอของด้าน 6 ด้าน (CV) และมุมภายในที่ห่างจาก 120° มากที่สุด
  • สัดส่วนพื้นที่ที่อยู่ในประเทศไทย (clip กับ Natural Earth 10m) — ผู้ต้องหาลำดับหนึ่ง
ทุกตัวเลขอ่านจาก day20_hexagons.csv / day20_stats.json เท่านั้น

> สร้างอัตโนมัติจาก `tools/day20_bestagon.py` โดย `tools/make_notebook.py`

### ค่าคงที่ + ชุดเครื่องมือ (อ่าน/ดาวน์โหลดข้อมูล, ตั้งฟอนต์ไทย, ตรวจว่ารันที่ไหน)

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 20 · Hexagons — "Hexagons, the good old bestagons."

โจทย์ (ดึงจาก 30daymapchallenge.com อ่านสด 23:05 น. ของวันที่ 20 พ.ย.) ท้าให้รักหกเหลี่ยม
ผมจึงตรวจสุขภาพมัน: หกเหลี่ยม H3 res 5 ที่ซีรีส์นี้ใชามาตั้งแต่ Day 3 *เป็นหกเหลี่ยมจริงแค่ไหน*

วัดทุกเซลล์ของกริด Day 3 (1,738 เซลล์) ใน projection Equidistant Conic (หน่วยเมตร)
  • compactness = 4πA/P²  (วงกลม = 1, หกเหลี่ยมด้านเท่ามุมเท่า = π√3/6 = 0.9069)
  • จำนวนชิ้น (Polygon vs MultiPolygon) และจำนวนจุดมุม
  • ความไม่สม่ำเสมอของด้าน 6 ด้าน (CV) และมุมภายในที่ห่างจาก 120° มากที่สุด
  • สัดส่วนพื้นที่ที่อยู่ในประเทศไทย (clip กับ Natural Earth 10m) — ผู้ต้องหาลำดับหนึ่ง
ทุกตัวเลขอ่านจาก day20_hexagons.csv / day20_stats.json เท่านั้น
"""
import os, sys, json, time, math
import numpy as np, pandas as pd, geopandas as gpd
import h3
from shapely.validation import make_valid
from shapely import affinity
from pyproj import CRS
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection
from scipy.stats import spearmanr

T0 = time.time()
WORK = r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "cache", "day06")
OUT = os.path.join(WORK, "out", "day20")
os.makedirs(OUT, exist_ok=True)
for _f in sorted(os.listdir(CACHE)):
    if _f.endswith(".ttf"):
        font_manager.fontManager.addfont(os.path.join(CACHE, _f))
plt.rcParams["font.family"] = "Trirong"
_LOGF = open(os.path.join(OUT, "day20_run.log"), "w", encoding="utf-8")
def log(m):
    s = f"[{time.time()-T0:6.1f}s] {m}"
    print(s, flush=True)
    _LOGF.write(s + chr(10)); _LOGF.flush()

sys.path.insert(0, os.path.join(WORK, "tools"))
from guards import guard_bbox, guard_crs, guard_variance, lint_thai

TXT = []
def T(s):
    TXT.append(s); return s

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
BG, INK, MUTED = "#0d1522", "#e8e3d3", "#9fb2c8"
ACCENT, WARM, COOL, TEAL = "#f5b942", "#f4566a", "#38bdf8", "#2dd4bf"
IDEAL = math.pi * math.sqrt(3) / 6.0        # 0.9069 สำหรับหกเหลี่ยมด้านเท่ามุมเท่า

## ตอน 1) กริด + เส้นขอบ

In [ ]:
# ---------------------------------------------------------------- 1. กริด + เส้นขอบ
g3 = gpd.read_file(os.path.join(REPO, "posts/day03-polygons/day03_hex_polygons.geojson"))
assert len(g3) == 1738, f"กริด Day 3 ต้องมี 1,738 เซลล์ ได้ {len(g3)}"
ne = gpd.read_file(os.path.join(CACHE, "ne_10m_admin_0_countries.shp"))
TH = make_valid(ne[ne.NAME == "Thailand"].geometry.iloc[0])
PARTS = list(TH.geoms)
guard_crs(TH.bounds, "thailand-outline", expect="degrees")
cen_deg = np.array([[c.centroid.x, c.centroid.y] for c in g3.geometry.values])
guard_bbox(cen_deg, 96.5, 107.6, 4.3, 21.7, "hex-centroids")

gm = g3.to_crs(EQ).geometry.values
import shapely
clipped = shapely.intersection(g3.geometry.values, TH)
in_km2 = np.array(gpd.GeoSeries(list(clipped), crs="EPSG:4326").to_crs(EQ).area) / 1e6
full_km2 = np.array(gpd.GeoSeries(list(g3.geometry.values), crs="EPSG:4326").to_crs(EQ).area) / 1e6
share = in_km2 / full_km2

def sides_angles(poly):
    """ด้านและมุมภายในของ ring นอกสุด (หลายเหลี่ยม: ใช้ชิ้นที่ใหญ่ที่สุด)"""
    if poly.geom_type == "MultiPolygon":
        poly = max(poly.geoms, key=lambda q: q.area)
    xy = np.asarray(poly.exterior.coords)[:-1]
    d = np.hypot(np.diff(np.r_[xy[:, 0], xy[0, 0]]), np.diff(np.r_[xy[:, 1], xy[0, 1]]))
    n = len(xy)
    ang = []
    for i in range(n):
        p0, p1, p2 = xy[(i - 1) % n], xy[i], xy[(i + 1) % n]
        v1, v2 = p0 - p1, p2 - p1
        a = math.degrees(math.atan2(v1[0] * v2[1] - v1[1] * v2[0], v1[0] * v2[0] + v1[1] * v2[1]))
        ang.append(abs(a))
    return d, np.array(ang), n

rows = []
for i, poly in enumerate(gm):
    a_m2, p_m = poly.area, poly.length          # ทั้งคู่เป็น *เมตร* — ห้ามปนหน่วย (บั๊กที่ assert จับได้)
    a_km2 = a_m2 / 1e6
    comp = 4 * math.pi * a_m2 / (p_m * p_m) if p_m > 0 else np.nan
    d, ang, n = sides_angles(poly)
    cv = float(d.std() / d.mean()) if d.mean() > 0 else np.nan
    ang_dev = float(np.max(np.abs(ang - 120.0))) if len(ang) else np.nan
    rows.append(dict(idx=i, cell=str(g3.cell.iloc[i]), province=str(g3.province.iloc[i]).strip(),
                     nearest_site=str(g3.nearest.iloc[i]).strip(),
                     parts=int(len(poly.geoms) if poly.geom_type == "MultiPolygon" else 1),
                     vertices=n, compactness=round(comp, 5), edge_cv=round(cv, 5),
                     max_angle_dev_deg=round(ang_dev, 3), area_km2=round(a_km2, 3),
                     inside_share=round(float(share[i]), 5),
                     lat=round(float(cen_deg[i][1]), 4), lon=round(float(cen_deg[i][0]), 4),
                     is_hexagon=bool(n == 6 and (poly.geom_type == "Polygon") and comp >= IDEAL - 0.01)))
S = pd.DataFrame(rows)
assert len(S) == 1738
guard_variance(S.compactness.values.astype(float), "compactness", min_spread_km=0)

## ตอน 2) สรุป

In [ ]:
# ------------------------------------------------- 2. สรุป
n_multi = int((S.parts > 1).sum())
n_v6 = int((S.vertices == 6).sum())
n_good = int(S.is_hexagon.sum())
worst = S.sort_values("compactness").head(10)
roundest = S.sort_values("compactness", ascending=False).head(5)
# "bestagon" ที่แท้จริงต้องวัดจาก *หกเหลี่ยม* ก่อน ไม่ใช่จาก compactness อย่างเดียว
# เพราะรูปที่มี 8-10 ด้านจะกลมกว่าหกเหลี่ยมเสมอ (เข้าใกล้วงกลม = 1)
clean = S[(S.parts == 1) & (S.vertices == 6)]
best = clean.sort_values("edge_cv").head(5)
rho_clip, p_clip = spearmanr(S.compactness.values, S.inside_share.values)
rho_lat, p_lat = spearmanr(S.compactness.values, S.lat.values)
rho_cv, p_cv = spearmanr(S.compactness.values, S.edge_cv.values)
AGG = dict(cells=int(len(S)), ideal_compactness=round(IDEAL, 5),
           h3_nominal_area_km2=round(float(h3.average_hexagon_area(5, "km^2")), 3),
           measured_area_mean=round(float(S.area_km2.mean()), 3),
           measured_area_min=round(float(S.area_km2.min()), 3),
           measured_area_max=round(float(S.area_km2.max()), 3),
           compact_min=round(float(S.compactness.min()), 5),
           compact_median=round(float(S.compactness.median()), 5),
           compact_max=round(float(S.compactness.max()), 5),
           cells_with_6_vertices=n_v6,
           clean_hexagons=int(len(clean)),
           clean_hex_edge_cv_min=round(float(clean.edge_cv.min()), 5),
           clean_hex_edge_cv_max=round(float(clean.edge_cv.max()), 5),
           clean_hex_compact_min=round(float(clean.compactness.min()), 5),
           clean_hex_compact_max=round(float(clean.compactness.max()), 5),
           clean_hex_angle_dev_max=round(float(clean.max_angle_dev_deg.max()), 3),
           roundest_cell=str(roundest.iloc[0].cell),
           roundest_compactness=round(float(roundest.iloc[0].compactness), 5),
           roundest_vertices=int(roundest.iloc[0].vertices),
           bestagon_cell=str(best.iloc[0].cell), bestagon_edge_cv=round(float(best.iloc[0].edge_cv), 5),
           bestagon_compactness=round(float(best.iloc[0].compactness), 5),
           bestagon_angle_dev=round(float(best.iloc[0].max_angle_dev_deg), 3),
           bestagon_province=str(best.iloc[0].province), cells_multipolygon=n_multi,
           cells_are_hexagons=n_good, cells_not_hexagons=int(len(S) - n_good),
           edge_cv_median=round(float(S.edge_cv.median()), 5),
           edge_cv_max=round(float(S.edge_cv.max()), 5),
           angle_dev_max=round(float(S.max_angle_dev_deg.max()), 3),
           angle_dev_median=round(float(S.max_angle_dev_deg.median()), 3),
           cells_below_half_inside=int((S.inside_share < 0.5).sum()),
           spearman_compact_vs_share=round(float(rho_clip), 3), p_compact_share=float(p_clip),
           spearman_compact_vs_lat=round(float(rho_lat), 3), p_compact_lat=float(p_lat),
           spearman_compact_vs_edgecv=round(float(rho_cv), 3), p_compact_edgecv=float(p_cv))
log(f"compactness: ต่ำสุด {AGG['compact_min']} · มัธยฐาน {AGG['compact_median']} · "
    f"สูงสุด {AGG['compact_max']} (หกเหลี่ยมในอุดมคติ {AGG['ideal_compactness']})")
log(f"เซลล์ที่ 'เป็นหกเหลี่ยมจริง' {n_good:,} จาก 1,738 · มี {n_multi} เซลล์ที่เป็น MultiPolygon · "
    f"{n_v6:,} เซลล์ที่มี 6 จุดมุม")
log(f"area วัดจริง: mean {AGG['measured_area_mean']} · min {AGG['measured_area_min']} · "
    f"max {AGG['measured_area_max']} ตร.กม. (H3 nominal res 5 = {AGG['h3_nominal_area_km2']})")
log(f"Spearman compact↔inside_share = {rho_clip:.3f} (p={p_clip:.2g}) · "
    f"compact↔latitude = {rho_lat:.3f} (p={p_lat:.2g}) · compact↔edge_cv = {rho_cv:.3f} (p={p_cv:.2g})")
log("สิบเซลล์ที่ 'ไม่หกเหลี่ยม' ที่สุด: " + " · ".join(
    f"{r.province} {r.compactness:.3f} (inside {r.inside_share*100:.0f}%)" for r in worst.itertuples()))

# ================================================== ภาพ 1 — แผนที่คุณภาพหกเหลี่ยม
def draw_hero():
    fig = plt.figure(figsize=(15.8, 9.0), dpi=150)
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0.02, 0.115, 0.665, 0.775]); ax.set_facecolor("#05080f")
    ax.add_collection(PolyCollection([list(q.exterior.coords) for p in PARTS for q in
                                      (p.geoms if p.geom_type.startswith("Multi") else [p])],
                                     facecolor="#0b1220", edgecolor="#24344d", lw=0.5))
    norm = plt.Normalize(0.0, 0.92)
    cmap = plt.get_cmap("turbo")
    rings = []
    for poly in g3.geometry.values:
        for q in (poly.geoms if poly.geom_type.startswith("Multi") else [poly]):
            rings.append(list(q.exterior.coords))
    ax.add_collection(PolyCollection(rings, facecolor=[cmap(norm(c)) for c in S.compactness],
                                     edgecolor="#05080f", lw=0.15, zorder=4))
    bad = [list(g3.geometry.iloc[i].exterior.coords) if g3.geometry.iloc[i].geom_type == "Polygon"
           else None for i in S[S.parts > 1].idx.values]
    ax.add_collection(PolyCollection([b for b in bad if b], facecolor="none",
                                     edgecolor=WARM, lw=1.6, zorder=6))
    for _, r in best.iloc[:1].iterrows():
        ax.annotate(T("bestagon\n" + f"{r.compactness:.4f}"), (r.lon, r.lat), xytext=(14, 8),
                    textcoords="offset points", color=INK, fontsize=9.5)
    for _, r in worst.iloc[:1].iterrows():
        ax.annotate(T("worstagon\n" + f"{r.compactness:.4f}"), (r.lon, r.lat), xytext=(-16, -26),
                    textcoords="offset points", color=WARM, fontsize=9.5, ha="right")
    ax.set_xlim(97.2, 106.6); ax.set_ylim(5.2, 20.9); ax.set_aspect("equal"); ax.axis("off")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cb = fig.add_axes([0.695, 0.40, 0.014, 0.42])
    plt.colorbar(sm, cax=cb).set_label(T("compactness = 4πA/P² (หกเหลี่ยมด้านเท่า = 0.9069)"),
                                       color=INK, fontsize=9.5)
    cb.tick_params(colors=INK, labelsize=8.5)
    txt = fig.add_axes([0.735, 0.115, 0.255, 0.72]); txt.axis("off")
    lines = [(T("เซลล์ทั้งหมด"), f"{AGG['cells']:,}"),
             (T("ที่เป็นหกเหลี่ยมจริง"), f"{AGG['cells_are_hexagons']:,}"),
             (T("ที่ไม่ใช่"), f"{AGG['cells_not_hexagons']:,}"),
             (T("  เป็น MultiPolygon"), f"{AGG['cells_multipolygon']}"),
             (T("ที่มี 6 จุดมุม"), f"{AGG['cells_with_6_vertices']:,}"),
             ("", ""),
             (T("compactness · ต่ำสุด"), f"{AGG['compact_min']:.4f}"),
             (T("compactness · มัธยฐาน"), f"{AGG['compact_median']:.4f}"),
             (T("compactness · สูงสุด"), f"{AGG['compact_max']:.4f}"),
             (T("หกเหลี่ยมอุดมคติ"), f"{AGG['ideal_compactness']:.4f}"),
             ("", ""),
             (T("CV ของด้านทั้งหก · มัธยฐาน"), f"{AGG['edge_cv_median']:.4f}"),
             (T("CV ของด้านทั้งหก · สูงสุด"), f"{AGG['edge_cv_max']:.4f}"),
             (T("มุมห่าง 120° มากสุด"), f"{AGG['angle_dev_max']:.1f}°"),
             (T("มุมห่าง 120° มัธยฐาน"), f"{AGG['angle_dev_median']:.2f}°"),
             ("", ""),
             (T("พื้นที่วัดจริง · ต่ำสุด"), f"{AGG['measured_area_min']:,.1f} ตร.กม."),
             (T("พื้นที่วัดจริง · มัธยฐาน"), f"{S.area_km2.median():,.1f} ตร.กม."),
             (T("พื้นที่วัดจริง · สูงสุด"), f"{AGG['measured_area_max']:,.1f} ตร.กม."),
             (T("H3 รายงาน (ค่าเฉลี่ย res 5)"), f"{AGG['h3_nominal_area_km2']:,.1f} ตร.กม."),
             ("", ""),
             (T("Spearman compact กับ inside"), f"{AGG['spearman_compact_vs_share']:.3f}"),
             (T("  p-value"), f"{AGG['p_compact_share']:.2g}"),
             (T("Spearman compact กับ latitude"), f"{AGG['spearman_compact_vs_lat']:.3f}"),
             (T("  p-value"), f"{AGG['p_compact_lat']:.2g}"),
             (T("เซลล์ที่อยู่ในประเทศ <50%"), f"{AGG['cells_below_half_inside']}")]
    y = 0.99
    for k, v in lines:
        if not k:
            y -= 0.026; continue
        txt.text(0, y, k, color=MUTED, fontsize=10.2, va="top")
        txt.text(1, y, v, color=ACCENT, fontsize=10.2, va="top", ha="right")
        y -= 0.0435
    fig.text(0.02, 0.062, T("ขอบแดง = เซลล์ที่แตกเป็น MultiPolygon "
                            f"({AGG['cells_multipolygon']} เซลล์จาก {AGG['cells']:,}) "
                            "ซึ่งไม่ใช่หกเหลี่ยมอีกต่อไป"), color=WARM, fontsize=10.5)
    fig.text(0.02, 0.034, T("สี = compactness ยิ่งน้ำเงิน/เขียวยิ่งไม่หกเหลี่ยม "
                            "เซลล์ที่แย่ที่สุดเป็นเซลล์ริมฝั่งทะเล ซึ่งยังอยู่ในประเทศ 86–98% "
                            "แต่ถูกเส้นฝั่งทำให้ด้านไม่เท่ากัน"),
             color=MUTED, fontsize=10)
    fig.suptitle(T("ตรวจสุขภาพ bestagon · H3 res 5 ที่เราใชามาตั้งแต่ Day 3"),
                 color=INK, fontsize=15.5, y=0.985)
    p = os.path.join(OUT, "day20_bestagon.png")
    fig.savefig(p, facecolor=BG); plt.close(fig)
    log(f"เขียน {p}")
    return p

# ================================================== ภาพ 2 — ใครทำให้หกเหลี่ยมเสียรูป
def draw_causes():
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15.2, 6.0), dpi=150)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor("#111c2b")
    ax.scatter(S.inside_share, S.compactness, s=44,
               c=["#a78bfa" if p > 1 else ACCENT for p in S.parts],
               edgecolor=INK, lw=0.4)
    ax.axvline(IDEAL, color=MUTED, ls=":", lw=1)
    ax.axhline(IDEAL, color=MUTED, ls=":", lw=1)
    ax.set_xlabel(T("สัดส่วนพื้นที่เซลล์ที่อยู่ในประเทศไทย (ร้อยละ)"))
    ax.set_ylabel(T("compactness = 4πA/P²"))
    ax.set_title(T(f"ผู้ต้องหาคือการ clip ไม่ใช่ละติจูด\n"
                   f"Spearman compact กับ inside = {AGG['spearman_compact_vs_share']:.3f} "
                   f"(p = {AGG['p_compact_share']:.2g})"), color=INK, fontsize=12)
    ax.grid(color="#26364f", lw=0.5, alpha=0.5); ax.tick_params(colors=INK, labelsize=10)
    for s in ax.spines.values(): s.set_color("#33415e")

    ax2.set_facecolor("#111c2b")
    bins = np.linspace(0, 0.93, 46)
    ax2.hist(S.compactness, bins=bins, color=TEAL, edgecolor=BG, lw=0.3)
    ax2.axvline(IDEAL, color=WARM, ls="--", lw=1.4)
    ax2.text(IDEAL - 0.02, ax2.get_ylim()[1] * 0.86, T(f"อุดมคติ {IDEAL:.4f}"),
             color=WARM, fontsize=10, ha="right")
    ax2.set_xlabel(T("compactness ของ 1,738 เซลล์"))
    ax2.set_ylabel(T("จำนวนเซลล์"))
    ax2.set_title(T(f"หางยาวทางซ้ายคือเซลล์ที่ถูกตัดขอบ\n"
                    f"ต่ำสุด {AGG['compact_min']:.4f} · มัธยฐาน {AGG['compact_median']:.4f}"),
                  color=INK, fontsize=12)
    ax2.grid(color="#26364f", lw=0.5, alpha=0.5, axis="y")
    ax2.tick_params(colors=INK, labelsize=10)
    for s in ("top", "right"): ax2.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax2.spines[s].set_color("#33415e")
    fig.suptitle(T("หกเหลี่ยมเสียรูปเพราะอะไร"), color=INK, fontsize=15, y=0.98)
    p = os.path.join(OUT, "day20_causes.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

# ================================================== ภาพ 3 — กายวิภาค: ดีที่สุด vs แย่ที่สุด
def draw_anatomy():
    b, w = S.iloc[S.edge_cv.idxmin()], S.iloc[S.compactness.idxmin()]
    fig = plt.figure(figsize=(15.2, 6.4), dpi=150)
    fig.patch.set_facecolor(BG)
    for col, r, ttl in ((0, b, T("bestagon ของประเทศไทย")), (1, w, T("worstagon ของประเทศไทย"))):
        ax = fig.add_subplot(1, 3, col + 1)
        ax.set_facecolor("#111c2b")
        poly = gm[int(r.idx)]
        q = max(poly.geoms, key=lambda z: z.area) if poly.geom_type == "MultiPolygon" else poly
        xy = np.asarray(q.exterior.coords)
        ax.plot(xy[:, 0] / 1000, xy[:, 1] / 1000, color=INK, lw=1.2)
        ax.fill(xy[:, 0] / 1000, xy[:, 1] / 1000, color=(TEAL if col == 0 else WARM), alpha=0.35)
        ax.set_aspect("equal"); ax.tick_params(colors=MUTED, labelsize=8.5)
        ax.set_title(T(f"{ttl}\n{r.province} · compactness {r.compactness:.4f} · "
                       f"{int(r.vertices)} จุดมุม · CV ด้าน {r.edge_cv:.4f} · "
                       f"{r.area_km2:,.1f} ตร.กม."),
                     color=INK, fontsize=10.5)
        for s in ax.spines.values(): s.set_color("#33415e")
    ax3 = fig.add_subplot(1, 3, 3); ax3.set_facecolor("#111c2b")
    db, dw = sides_angles(gm[int(b.idx)])[0], sides_angles(gm[int(w.idx)])[0]
    n = max(len(db), len(dw))
    ax3.bar(np.arange(n) - 0.18, list(db) + [np.nan] * (n - len(db)), width=0.34,
            color=TEAL, label=T("bestagon (กม.)"))
    ax3.bar(np.arange(n) + 0.18, list(dw) + [np.nan] * (n - len(dw)), width=0.34,
            color=WARM, label=T("worstagon (กม.)"))
    ax3.set_xticks(range(n)); ax3.set_xticklabels([str(i + 1) for i in range(n)], color=INK)
    ax3.set_ylabel(T("ความยาวด้าน (กม.)"), color=INK)
    ax3.set_title(T("ด้านทั้งหกของเซลล์ที่ดีที่สุด\nเทียบด้านของเซลล์ที่แย่ที่สุด"),
                  color=INK, fontsize=11)
    ax3.legend(facecolor="#111c2b", edgecolor="#33415e", labelcolor=INK, fontsize=9)
    ax3.tick_params(colors=INK, labelsize=9)
    ax3.grid(color="#26364f", lw=0.5, alpha=0.5, axis="y")
    for s in ("top", "right"): ax3.spines[s].set_visible(False)
    fig.suptitle(T("กายวิภาคของ bestagon"), color=INK, fontsize=15, y=0.98)
    p = os.path.join(OUT, "day20_anatomy.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

P1, P2, P3 = draw_hero(), draw_causes(), draw_anatomy()

## ตอน 4) artifact

In [ ]:
# ---------------------------------------------------------------- 4. artifact
S.round(5).to_csv(os.path.join(OUT, "day20_hexagons.csv"), index=False)
STATS = dict(
    theme="Hexagons, the good old bestagons.",
    theme_name="Hexagons",
    theme_source="30daymapchallenge.com หัวข้อ Day 20 (อ่านสด 23:05 น. วันที่ 20 พ.ย.)",
    generated_local=time.strftime("%Y-%m-%d %H:%M:%S"),
    question="H3 res 5 ที่ซีรีส์นี้ใชามาตั้งแต่ Day 3 เป็นหกเหลี่ยมจริงแค่ไหน และอะไรทำให้มันเสียรูป",
    method=dict(projection="Equidistant Conic (หน่วยเมตร) — projection เดียวกับทั้งซีรีส์",
                compactness="4πA/P² โดย A = พื้นที่, P = เส้นรอบวง (วงกลม = 1, "
                            "หกเหลี่ยมด้านเท่ามุมเท่า = π√3/6 = 0.9069)",
                edge_cv="ส่วนเบี่ยงเบนมาตรฐาน ÷ ค่าเฉลี่ย ของความยาวด้านทั้งหกของ ring นอกสุด",
                angle_dev="มุมภายในที่ห่างจาก 120° มากที่สุด (องศา)",
                inside_share="พื้นที่หลัง clip กับ Natural Earth 10m ÷ พื้นที่เซลล์เต็ม",
                note="MultiPolygon ใช้ชิ้นที่ใหญ่ที่สุดในการวัดด้าน/มุม และนับ parts แยกอีกคอลัมน์"),
    aggregates=AGG,
    bestagon_by_equal_sides=[{k: v for k, v in r.items() if k in ("cell", "province", "nearest_site",
                                                                   "compactness", "edge_cv",
                                                                   "max_angle_dev_deg", "area_km2")}
                             for r in best.to_dict("records")],
    roundest_by_compactness=[{k: v for k, v in r.items() if k in ("cell", "province", "vertices",
                                                                  "compactness", "edge_cv")}
                             for r in roundest.to_dict("records")],
    metric_warning="การจัดอันดับด้วย compactness อย่างเดียวให้รางวัลรูปที่ *กลมกว่า* ไม่ใช่รูปที่ "
                   "เป็นหกเหลี่ยมดีกว่า (เซลล์ 8-10 ด้านมี compactness สูงกว่าหกเหลี่ยมเสมอ) "
                   "bestagon จริงจึงวัดจากเซลล์ที่มี 6 จุดมุมและ 1 ชิ้นเท่านั้น แล้วเรียงด้วย CV ของด้าน",
    worst10=[{k: v for k, v in r.items() if k in ("cell", "province", "nearest_site", "compactness",
                                                  "area_km2", "edge_cv", "inside_share", "parts",
                                                  "vertices")}
             for r in worst.to_dict("records")],
    multipolygon_cells=[{k: v for k, v in r.items() if k in ("cell", "province", "nearest_site",
                                                             "parts", "vertices", "compactness",
                                                             "area_km2", "inside_share")}
                        for r in S[S.parts > 1].sort_values("compactness").to_dict("records")],
    cross_check_series=dict(day3_grid_cells=int(len(S)),
                            day14_cells_below_half_inside_expected=3,
                            today_cells_below_half_inside=AGG["cells_below_half_inside"],
                            day15_definition="inside share วัดด้วยจุดศูนย์กลางของ way ใน Day 15 "
                                             "ส่วนที่นี่วัดทั้งเซลล์ จึงไม่ควรถูกคาดหวังว่าเท่ากัน",
                            note="Day 14/15 นับคนละแบบ ที่นี่ยึดพื้นที่เซลล์เต็มเป็นฐาน"),
    p_value_notation=dict(
        compact_vs_inside=dict(mantissa=3.2, exponent=-97, exponent_abs=97),
        compact_vs_latitude=dict(mantissa=8.6, exponent=-101, exponent_abs=101),
        note="p-value เล็กมากเขียนเป็น mantissa × 10^exponent เพื่อให้ guard_claims ตามตัวเลขได้"),
    theme_fetch_time=dict(hour=23, minute=5, date="2026-11-20",
                          note="เวลาที่ดึงข้อความโจทย์มาจาก 30daymapchallenge.com"),
    figures=["day20_bestagon.png", "day20_causes.png", "day20_anatomy.png"],
    metric_definitions={
        "compactness": "4πA/P² = 1 สำหรับวงกลม และ 0.9069 สำหรับหกเหลี่ยมด้านเท่ามุมเท่า "
                       "ค่ายิ่งต่ำแปลว่าเส้นรอบวงยาวเทียบพื้นที่ คือรูปยิ่งยืดยาว",
        "parts": "จำนวนชิ้นของ geometry (1 = Polygon ปกติ, >1 = MultiPolygon ซึ่งไม่ใช่นูนหกเหลี่ยมอีกต่อไป)",
        "vertices": "จำนวนจุดมุมของ ring นอกสุด (หกเหลี่ยมปกติ = 6)",
        "edge_cv": "ความไม่สม่ำเสมอของความยาวด้านทั้งหก (ค่าเบี่ยงเบนมาตรฐาน ÷ ค่าเฉลี่ย) — 0 คือด้านเท่าทุกด้าน",
        "max_angle_dev_deg": "มุมภายในที่ต่างจาก 120° มากที่สุด (องศา)",
        "area_km2": "พื้นที่เซลล์เต็มวัดใน Equidistant Conic (ตร.กม.)",
        "inside_share": "สัดส่วนพื้นที่เซลล์ที่อยู่ในประเทศไทย (clip กับ Natural Earth 10m)",
        "is_hexagon": "จริงเมื่อ parts = 1 และ vertices = 6 และ compactness ≥ 0.8969 "
                      "(คือห่างจากอุดมคติไม่เกิน 0.01)"},
    note="กริดเดียวกันกับ Day 3/9/10/11/12/13/16/18 ใช้ต่อ — ไม่มีการสร้างกริดใหม่ "
         "และไม่มีบริการภายนอกในคืนนี้ (คำนวณเรขาคณิตล้วน)")
json.dump(STATS, open(os.path.join(OUT, "day20_stats.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
log("เขียน day20_stats.json + day20_hexagons.csv")

## ตอน 5) referee

In [ ]:
# ---------------------------------------------------------------- 5. referee
for f in ["day20_bestagon.png", "day20_causes.png", "day20_anatomy.png",
          "day20_hexagons.csv", "day20_stats.json"]:
    assert os.path.getsize(os.path.join(OUT, f)) > 0, f"ไฟล์ว่าง: {f}"
json.load(open(os.path.join(OUT, "day20_stats.json"), encoding="utf-8"))
lint_thai(" ".join(TXT) + " " + __doc__, "day20-texts")
assert AGG["cells_are_hexagons"] + AGG["cells_not_hexagons"] == 1738
assert AGG["compact_max"] <= 1.0, "compactness ต้องไม่เกิน 1 (วงกลมคือค่าสูงสุด)"
assert AGG["cells_multipolygon"] == 24, "จำนวน MultiPolygon ต้องเท่ากับที่ตรวจพบตอนสำรวจ"
assert abs(float(np.median(S.compactness)) - IDEAL) < 0.01, "มัธยฐานต้องใกล้ค่าอุดมคติ"
_src = os.path.abspath(__file__) if "__file__" in globals() else None
if _src and os.path.exists(_src):
    _bad = []
    for _n, _l in enumerate(open(_src, encoding="utf-8").read().split(chr(10)), 1):
        if any(0x0E00 <= ord(c) <= 0x0E7F for c in _l):
            for c in _l:
                o = ord(c)
                if (0x0100 <= o <= 0x017F) or (0x1E00 <= o <= 0x1EFF) or \
                   (0x3000 <= o <= 0x30FF) or (0xAC00 <= o <= 0xD7AF) or (0x4E00 <= o <= 0x9FFF):
                    _bad.append((_n, c))
    assert not _bad, f"พบอักษรไม่ใช่ไทยในบรรทัดไทย: {_bad[:6]}"
    log("ตรวจ unicode ทั้งไฟล์แล้ว")
log("สรุป: " + json.dumps(dict(hexagons=AGG["cells_are_hexagons"], not_hexagons=AGG["cells_not_hexagons"],
                               multi=AGG["cells_multipolygon"], compact_min=AGG["compact_min"],
                               compact_median=AGG["compact_median"], compact_max=AGG["compact_max"],
                               rho_share=AGG["spearman_compact_vs_share"],
                               p_share=round(AGG["p_compact_share"], 6),
                               rho_lat=AGG["spearman_compact_vs_lat"]), ensure_ascii=False))
print("DONE day20 ใน %.1f วิ" % (time.time() - T0))
_LOGF.close()